# Khaledoon — OmniVoice Unified Worker
عامل توليد واحد لكل الحكايات: كارما والراوي مزوّدان منفصلان داخل نفس المحرك.
العامل **لا يقرر ماذا يُولَّد**؛ ينفّذ فقط الطلبات التي ضغطتَ توليدها من صفحة المعاينة، مقطعًا واحدًا في كل مرة، ويرفع الناتج إلى التخزين الخاص للمراجعة. لا نشر تلقائي.

**مرة واحدة فقط:** احفظ `KHALEDOON_WORKER_TOKEN` في Colab Secrets (فعّل Notebook access). بعدها لا يطلب الدفتر أي كلمة سر. نص كارما مثبّت داخل الدفتر ومحمي ببصمة SHA-256. إعدادات كارما (36/3.0/0.90) والراوي (48/3.0/1.05) ثابتة داخل المحرك.
الراوي صوت جاهز من مكتبة VoiceTut باسم `Karim` (spk_04_male)؛ بيتحمّل تلقائيًا ومش محتاج ملف ولا Drive ولا سر.

In [ ]:
%pip -q install git+https://github.com/k2-fsa/OmniVoice.git soundfile requests numpy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/engmohamedgamalsaad-oss/nourdeen-october-1973.git'
ENGINE_BRANCH = 'main'
if not os.path.isdir('/content/khaledoon'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', ENGINE_BRANCH, REPO_URL, '/content/khaledoon'], check=True)
sys.path.insert(0, '/content/khaledoon/scripts')
import omnivoice_engine
print('Engine loaded from', omnivoice_engine.__file__)

In [ ]:
import uuid
from google.colab import userdata

WORKER_URL = 'https://ypwprddkscnfblhftkeb.supabase.co/functions/v1/story-audio-worker'
KARMA_REF_AUDIO_PATH = '/content/khaledoon/assets/audio/science/magdi-yacoub/ar/000.mp3'  # من الريبو نفسه، بدون Drive

def secret(name):
    try:
        return str(userdata.get(name) or '').strip()
    except Exception:
        return ''

WORKER_TOKEN = secret('KHALEDOON_WORKER_TOKEN')
KARMA_REF_TEXT = 'أنا كارما، ولما سمعت حكاية الدكتور مجدي يعقوب، فكرت في حاجة مهمة: إزاي طفل بيحب يساعد الناس ممكن يكبر ويبقى طبيبًا ينقذ حياة أطفال كتير؟'  # مقفول بـ SHA-256 داخل المحرك
NARRATOR_LIBRARY_VOICE = 'karim'
if not WORKER_TOKEN:
    raise ValueError('احفظ KHALEDOON_WORKER_TOKEN في Colab Secrets (مرة واحدة) وفعّل Notebook access.')
WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]
print('Config ready:', WORKER_ID)

In [ ]:
import torch
from omnivoice import OmniVoice
from omnivoice_engine import MODEL_ID
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if device == 'cpu':
    print('تحذير: الجهاز CPU. اختر Runtime ثم Change runtime type ثم GPU.')
voice_model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
print('Model loaded on', device)

In [ ]:
import os
import uuid
from omnivoice_engine import KarmaProvider, NarratorProvider, ProviderRegistry, WorkerClient

registry = ProviderRegistry()
registry.register(KarmaProvider(KARMA_REF_AUDIO_PATH, KARMA_REF_TEXT))
from huggingface_hub import snapshot_download
refs_dir = os.path.join(snapshot_download('mohammedaly22/VoiceTut-TTS', allow_patterns=['reference_speakers/*']), 'reference_speakers')
registry.register(NarratorProvider.from_library(refs_dir, NARRATOR_LIBRARY_VOICE))
print('Providers: karma + narrator (' + NARRATOR_LIBRARY_VOICE + ')')
if not globals().get('WORKER_ID'):
    WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]
    print('Worker ID regenerated:', WORKER_ID)
client = WorkerClient(WORKER_URL, WORKER_TOKEN, WORKER_ID)

In [ ]:
# يعمل حتى توقف الخلية يدويًا. أثناء الانتظار يظهر العامل «متصلًا» في صفحة المعاينة.
from omnivoice_engine import run_worker
run_worker(client, voice_model, registry, poll_seconds=20)